# Window functions

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql-08-window-functions.ipynb)

A **window function** computes a value over a set of rows related to the current row, **without collapsing them**.
A `GROUP BY` returns one row per group. A window keeps every row and adds a column, so you can show each order next
to its customer's running total, or next to its rank.

The syntax is `function() OVER (PARTITION BY ... ORDER BY ... frame)`. `PARTITION BY` restarts the window for each
group, `ORDER BY` sets the sequence, and the frame chooses which rows are included. This chapter builds up those
parts one at a time and ends with practice questions and answers.

## Setup

Install PySpark, start a Spark session, and create the example tables.

In [ ]:
!pip install -q pyspark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW customers AS
SELECT * FROM VALUES
  (1, 'Acme Corp',    'North', DATE '2022-01-05'),
  (2, 'Beta LLC',     'North', DATE '2022-02-10'),
  (3, 'Cyan Inc',     'South', DATE '2022-03-15'),
  (4, 'Delta Co',     'South', DATE '2022-04-20'),
  (5, 'Echo Ltd',     'West',  DATE '2022-05-25'),
  (6, 'Foxtrot GmbH', 'West',  DATE '2022-06-30'),
  (7, 'Gamma SA',     'East',  DATE '2023-01-10'),
  (8, 'Hotel BV',     'East',  DATE '2023-02-15')
AS customers(customer_id, name, region, signup_date)
""")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW orders AS
SELECT * FROM VALUES
  (1,  1, 'completed', 100, DATE '2023-01-05'),
  (2,  1, 'completed', 200, DATE '2023-02-10'),
  (3,  1, 'completed', 150, DATE '2023-03-15'),
  (4,  2, 'completed', 300, DATE '2023-01-20'),
  (5,  2, 'cancelled', 250, DATE '2023-02-05'),
  (6,  2, 'completed', 400, DATE '2023-03-20'),
  (7,  3, 'completed', 120, DATE '2023-01-25'),
  (8,  3, 'completed', 180, DATE '2023-02-28'),
  (9,  4, 'pending',   220, DATE '2023-02-14'),
  (10, 4, 'completed', 320, DATE '2023-03-30'),
  (11, 5, 'completed', 500, DATE '2023-01-02'),
  (12, 5, 'completed', 450, DATE '2023-03-05'),
  (13, 6, 'completed', 275, DATE '2023-02-18'),
  (14, 6, 'cancelled', 325, DATE '2023-03-25')
AS orders(order_id, customer_id, status, amount, order_date)
""")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW pay AS
SELECT * FROM VALUES ('Alice', 150), ('Bob', 120), ('Carol', 120), ('Dave', 95)
AS pay(name, salary)
""")

## 1. A window keeps every row

Compare a `GROUP BY` with a window. The grouped query has one row per customer. The window version has one row per
order, and each row also carries its customer's total.

In [ ]:
spark.sql("""
SELECT customer_id, COUNT(*) AS orders FROM orders GROUP BY customer_id ORDER BY customer_id
""").show(3)

spark.sql("""
SELECT order_id, customer_id, amount,
       SUM(amount) OVER (PARTITION BY customer_id) AS customer_total
FROM orders
ORDER BY customer_id, order_id
LIMIT 4
""").show()

## 2. Ranking: ROW_NUMBER, RANK and DENSE_RANK

All three number the rows in a window. They differ only in how they treat ties.

- `ROW_NUMBER()` gives every row a different number. Ties are broken arbitrarily unless you add a tie-breaker.
- `RANK()` gives tied rows the same number, then skips ahead.
- `DENSE_RANK()` gives tied rows the same number, with no gap.

Bob and Carol tie at 120.

In [ ]:
spark.sql("""
SELECT name, salary,
       ROW_NUMBER() OVER (ORDER BY salary DESC, name) AS row_number,
       RANK()       OVER (ORDER BY salary DESC)       AS rank,
       DENSE_RANK() OVER (ORDER BY salary DESC)       AS dense_rank
FROM pay
ORDER BY salary DESC, name
""").show()

## 3. Running totals with PARTITION BY and ORDER BY

`PARTITION BY` restarts the window for each customer. `ORDER BY` sets the order of the running total. The
`order_id` tie-breaker keeps the result deterministic.

In [ ]:
spark.sql("""
SELECT customer_id, order_id, amount,
       SUM(amount) OVER (PARTITION BY customer_id ORDER BY order_date, order_id) AS running_total
FROM orders
WHERE status = 'completed' AND customer_id IN (1, 2)
ORDER BY customer_id, order_date
""").show()

## 4. Frames: ROWS versus RANGE

When rows tie on the `ORDER BY` value, the frame decides what each row sees. `ROWS` counts physical rows, so tied
rows get different running totals. `RANGE` includes every peer, so tied rows share a total. Be explicit whenever ties
can occur.

In [ ]:
spark.sql("""
SELECT name, salary,
       SUM(salary) OVER (ORDER BY salary, name
                         ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS rows_total,
       SUM(salary) OVER (ORDER BY salary
                         RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS range_total
FROM pay
ORDER BY salary, name
""").show()

## 5. LAG: the previous row

`LAG(x)` returns the value of `x` from the previous row in the window order. Aggregate to one row per period
first, then apply `LAG`. Applied to raw orders, it would return the previous order, not the previous month.

In [ ]:
spark.sql("""
WITH monthly AS (
  SELECT date_format(order_date, 'yyyy-MM') AS month, SUM(amount) AS revenue
  FROM orders
  WHERE status = 'completed'
  GROUP BY date_format(order_date, 'yyyy-MM')
)
SELECT month, revenue,
       LAG(revenue) OVER (ORDER BY month) AS previous_revenue,
       ROUND(100.0 * (revenue - LAG(revenue) OVER (ORDER BY month))
             / LAG(revenue) OVER (ORDER BY month), 2) AS growth_pct
FROM monthly
ORDER BY month
""").show()

## 6. Share of total

`SUM(SUM(amount)) OVER ()` nests a window over an aggregate. The inner `SUM` totals each customer. The outer window
totals those totals across the whole result, so each customer's share needs no second query.

In [ ]:
spark.sql("""
SELECT customer_id,
       SUM(amount) AS total,
       ROUND(100.0 * SUM(amount) / SUM(SUM(amount)) OVER (), 2) AS pct_of_total
FROM orders
WHERE status = 'completed'
GROUP BY customer_id
ORDER BY customer_id
""").show()

## 7. Filtering on a window result

A window cannot appear in `WHERE`, because windows are computed after `WHERE`. Spark SQL has no `QUALIFY` clause, so
filter the window result in an outer query. This example keeps the largest order per customer:

In [ ]:
spark.sql("""
SELECT customer_id, order_id, amount
FROM (
  SELECT customer_id, order_id, amount,
         ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY amount DESC, order_id) AS rn
  FROM orders
)
WHERE rn = 1
ORDER BY customer_id
""").show()

## More window functions

`LEAD` is the mirror of `LAG`: it reads the next row. A **moving average** uses a fixed frame of rows, here the current
order and the two before it. `FIRST_VALUE` and `LAST_VALUE` read the first and last row of the frame. Note the frame in
the `LAST_VALUE` example: with the default frame, the last value is the current row, so the full frame must be spelled out.
`NTILE(n)` splits the rows into `n` buckets of nearly equal size. A `WINDOW` clause names a window once so several
functions can share it.

In [ ]:
spark.sql("""
SELECT customer_id, order_date, amount,
       LEAD(amount) OVER (PARTITION BY customer_id ORDER BY order_date) AS next_amount
FROM orders
WHERE customer_id IN (1, 2)
ORDER BY customer_id, order_date
""").show()

spark.sql("""
SELECT customer_id, order_date, amount,
       ROUND(AVG(amount) OVER (PARTITION BY customer_id ORDER BY order_date
                               ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 1) AS moving_avg_3
FROM orders
WHERE customer_id = 1
ORDER BY order_date
""").show()

spark.sql("""
SELECT customer_id, order_date, amount,
       FIRST_VALUE(amount) OVER (PARTITION BY customer_id ORDER BY order_date) AS first_amount,
       LAST_VALUE(amount)  OVER (PARTITION BY customer_id ORDER BY order_date
                                 ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) AS last_amount
FROM orders
WHERE customer_id = 2
ORDER BY order_date
""").show()

spark.sql("""
SELECT order_id, amount, NTILE(4) OVER (ORDER BY amount) AS quartile
FROM orders
ORDER BY amount
""").show()

spark.sql("""
SELECT customer_id, order_id,
       ROW_NUMBER() OVER w AS nth_order,
       SUM(amount) OVER w AS running_total
FROM orders
WHERE customer_id = 1
WINDOW w AS (PARTITION BY customer_id ORDER BY order_date, order_id)
ORDER BY customer_id, nth_order
""").show()

The `WINDOW` clause names a window once, so `ROW_NUMBER` and `SUM` share it. It comes after `WHERE`, `GROUP BY` and
`HAVING`, and before `ORDER BY`. Filtering with `WHERE` before the window changes the result, just like filtering
before a `GROUP BY`.

## Interview notes

- **The default frame.** With `ORDER BY` and no frame, the frame is `RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT
  ROW`. Rows that tie on the order value share one running total. Add `ROWS` when each row should count alone.
- **ROW_NUMBER vs RANK vs DENSE_RANK.** Use `ROW_NUMBER` for exactly one row per group, with a unique tie-breaker.
  Use `RANK` or `DENSE_RANK` when ties should all appear.
- **A window cannot appear in WHERE.** Compute it in a subquery or CTE, then filter.
- **Cost.** A window with a large frame or no `PARTITION BY` moves all the rows through one worker. Partition the
  window whenever the logic allows.

## Summary

- A window keeps every row. A `GROUP BY` collapses them.
- Give `ROW_NUMBER` a unique tie-breaker, and make the frame explicit when ties can occur.
- `LAG` and `LEAD` look at neighbouring rows. Aggregate to the right period first.
- Filter on a window result in an outer query.

## Practice questions

1. Rank the customers by their total completed amount, highest first. Show `customer_id`, the total and the rank,
   sorted by rank.
2. For customer 1, show each order with the amount of the previous order. Sort by `order_id`. The first order has no previous order.
3. For each customer, show the amount of their second-largest order, counting every status. Sort by `customer_id`.

## Answers

### Question 1

In [ ]:
q = """
WITH totals AS (
  SELECT customer_id, SUM(amount) AS total
  FROM orders
  WHERE status = 'completed'
  GROUP BY customer_id
)
SELECT customer_id, total, RANK() OVER (ORDER BY total DESC) AS rank
FROM totals
ORDER BY rank
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(5, 950, 1), (2, 700, 2), (1, 450, 3), (4, 320, 4), (3, 300, 5), (6, 275, 6)]

### Question 2

In [ ]:
q = """
SELECT order_id, amount,
       LAG(amount) OVER (ORDER BY order_date) AS previous_amount
FROM orders
WHERE customer_id = 1
ORDER BY order_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(1, 100, None), (2, 200, 100), (3, 150, 200)]

### Question 3

In [ ]:
q = """
SELECT customer_id, amount AS second_largest
FROM (
  SELECT customer_id, amount,
         ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY amount DESC) AS rn
  FROM orders
)
WHERE rn = 2
ORDER BY customer_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(1, 150), (2, 300), (3, 120), (4, 220), (5, 450), (6, 275)]